# Causal Modeling

Question: What is the effect of having top actors on a movie’s chance of success?

In [7]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
import statsmodels.api as sm
from collections import Counter
import ast

df = pd.read_csv('../data/movies.csv')

Defining Success (top 25% revenue)

In [8]:
threshold = df["revenue"].quantile(0.75)
df["success"] = (df["revenue"] >= threshold).astype(int)

Release Date Parsing

In [9]:
df["release_date"] = pd.to_datetime(df["release_date"])
df["release_month"] = df["release_date"].dt.month
df["release_year"] = df["release_date"].dt.year

Actor Counts

In [10]:
df["cast_parsed"] = df["cast"].apply(ast.literal_eval)
df["actor_names"] = df["cast_parsed"].apply(lambda x: [d["name"] for d in x])

all_actors = [actor for sublist in df["actor_names"] for actor in sublist]

actor_counts = Counter(all_actors)
common_20_actors = actor_counts.most_common(20)
top_20_actors = [name[0] for name in actor_counts.most_common(20)]
df["num_top_actors"] = df["actor_names"].apply(
    lambda actors: sum(1 for actor in actors if actor in top_20_actors)
)

df['has_top_actor'] = (df['num_top_actors'] > 0).astype(int)

Genre Names

In [11]:
# One-Hot Encoding the Genre Names

df["genres_parsed"] = df["genres"].apply(ast.literal_eval)
df["genre_names"] = df["genres_parsed"].apply(lambda x: [d["name"] for d in x])
df.explode("genre_names")

mlb = MultiLabelBinarizer()
genres_encoded = mlb.fit_transform(df["genre_names"])
genre_df = pd.DataFrame(genres_encoded, columns=mlb.classes_, index=df.index)
df = pd.concat([df, genre_df], axis=1).dropna()

#### Variables <br>
Treatment: movie has top actor<br><br>
Outcome: movie success<br><br>
Confounders: budget, genre, release timing<br><br>

In [12]:
covariates = ['budget', 'release_year', 'release_month'] + list(mlb.classes_)
X = df[covariates]
X = sm.add_constant(X)
y = df['success']

X['treatment'] = df['has_top_actor']

model = sm.Logit(y, X)
result = model.fit()
print(result.summary())

margins = result.get_margeff()
print(margins.summary())

         Current function value: 0.399816
         Iterations: 35
                           Logit Regression Results                           
Dep. Variable:                success   No. Observations:                 1494
Model:                          Logit   Df Residuals:                     1469
Method:                           MLE   Df Model:                           24
Date:                Fri, 27 Mar 2026   Pseudo R-squ.:                  0.4106
Time:                        14:10:16   Log-Likelihood:                -597.33
converged:                      False   LL-Null:                       -1013.5
Covariance Type:            nonrobust   LLR p-value:                2.963e-160
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
const              84.3479     16.074      5.247      0.000      52.843     115.853
budget           5.165e-08   3.34e-09     15.454  

/Users/raimaazrafiislam/opt/miniconda3/envs/newenv/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


### Interpreting the treatment variable (a movie having top actors):

- treatment coef = 0.0503
- p-value = 0.041

Our positive coefficient signals that movies with a top actor **are** more likely to be successful.

Because this is logistic regression, our coefficient is in log-odds. We can convert to  odds ratio by calculating:
#### e^(0.0503) ≈ 1.052

This means that having a top-20 actor increases the odds of a movie being financially successful by about **5.2%**, holding budget, release timing, and genre constant.

### Interpreting the Budget variable:

- coef = 5.165e-08
- p-value = 0.000

Our p-value being lower than 0.001 signals that higher-budget movies are significantly more likely to be financially successful.

### Interpreting the Time variable:

#### Release Year:

- coef = -0.0431
- p-value = 0.000

Movies released in later years are slightly less likely to fall in the top 25% revenue threshold.

#### Release Month:

- coef: 0.0432
- p-value: 0.053

Movies released later in the year may have slightly higher chances of success, but only very weakly.

### Interpreting the Genre variables:

The only genres that were statistically significant (p < 0.05) were Crime, Drama, Science Fiction, Thriller, and Western.

#### Crime
e^(−0.608) ≈ 0.54 <br>
- Crime movies have about 46% lower odds of being top-revenue films compared with the baseline genre.

#### Drama
e^(-0.4461) ≈ 0.64 <br>
- Drama movies have about 36% lower odds of being top-revenue films compared with the baseline genre.

#### Science Fiction
e^(-0.6159) ≈ 0.54 <br>
- Science Fiction movies have about 46% lower odds of being top-revenue films compared with the baseline genre.

#### Thriller
e^(-0.4169) ≈ 0.66 <br>
- Thriller movies have about 34% lower odds of being top-revenue films compared with the baseline genre.

#### Western
e^(-2.0697) ≈ 0.13 <br>
- Western have about 87% lower odds of being top-revenue films compared with the baseline genre.

## Summary

#### Original Question: Does casting a top actor increase the probability that a movie becomes a financial success?

#### Answer: The logistic regression suggests that having a top actor has a small but statistically significant positive effect on movie success. After controlling for budget, release timing, and genre, movies with top actors have approximately 5% higher odds of being in the top revenue quartile. However, budget is a  stronger predictor of success. Additionally, the model did not fully converge -- results should be interpreted with caution.